## Imports & Config


In [18]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import ConvNeXt_Base_Weights
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/data/home/<user>/Astana_satellite_images"
CSV_PATH = os.path.join(DATA_DIR, "Astana_satellite_metadata.csv")

IMG_SIZE = 384
BATCH_SIZE = 16
EPOCHS = 80
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

def set_seed(seed):
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda | PyTorch: 2.14.0a0+git1a0a3c9
GPU: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition


## Dataset & Data Prep



In [19]:
class SatelliteHeightDataset(Dataset):
  def __init__(self, csv_df, image_dir, transform=None):
      self.df = csv_df.reset_index(drop=True)
      self.image_dir = image_dir
      self.transform = transform

  def __len__(self):
      return len(self.df)

  def __getitem__(self, idx):
      row = self.df.iloc[idx]
      img = Image.open(os.path.join(self.image_dir, row["filename"])).convert("RGB")
      if self.transform:
          img = self.transform(img)
      height = float(row["height"])
      return img, height, height

df = pd.read_csv(CSV_PATH)
df = df[df["status"] == "OK"].copy()
df = df[df["filename"].apply(lambda f: os.path.exists(os.path.join(DATA_DIR, f)))].copy()
print(f"Samples: {len(df)}, Height: {df['height'].min():.1f}m – {df['height'].max():.1f}m, Median: {df['height'].median():.1f}m")

train_df, temp_df = train_test_split(df, test_size=0.1, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(45),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])

val_test_transform = transforms.Compose([
  transforms.Resize((IMG_SIZE, IMG_SIZE)),
  transforms.ToTensor(),
  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

weights = []
for _, row in train_df.iterrows():
  if row["height"] > 20:
      weights.append(3.0)
  elif row["height"] > 10:
      weights.append(2.0)
  else:
      weights.append(1.0)

sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

train_loader = DataLoader(SatelliteHeightDataset(train_df, DATA_DIR, train_transform),
                        batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(SatelliteHeightDataset(val_df, DATA_DIR, val_test_transform),
                      batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(SatelliteHeightDataset(test_df, DATA_DIR, val_test_transform),
                       batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


Samples: 5998, Height: 2.5m – 307.1m, Median: 6.4m
Train: 5398, Val: 300, Test: 300


## 



In [20]:
backbone = models.convnext_base(weights=ConvNeXt_Base_Weights.DEFAULT)
x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
  num_features = backbone.features(x).shape[1]
print(f"num_features: {num_features}")

class ConvNeXtRegression(nn.Module):
  def __init__(self, backbone, num_features):
      super().__init__()
      self.features = backbone.features
      self.head = nn.Sequential(
          nn.AdaptiveAvgPool2d(1),
          nn.Flatten(),
          nn.Dropout(0.3),
          nn.Linear(num_features, 512),
          nn.GELU(),
          nn.Dropout(0.2),
          nn.Linear(512, 128),
          nn.GELU(),
          nn.Dropout(0.1),
          nn.Linear(128, 1),
      )
  def forward(self, x):
      return self.head(self.features(x)).squeeze(-1)

model = ConvNeXtRegression(backbone, num_features).to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

num_features: 1024
Params: 88,155,009


## 



In [21]:
criterion = nn.HuberLoss(delta=3.0, reduction="mean")
FREEZE_BACKBONE_EPOCHS = 5

def set_trainable(module, trainable):
  for p in module.parameters():
      p.requires_grad = trainable

set_trainable(model.features, False)

optimizer = torch.optim.AdamW([
  {"params": model.head.parameters(), "lr": LR},
  {"params": model.features.parameters(), "lr": LR / 10},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

## 



In [22]:
print("="*60 + "\nTRAINING\n" + "="*60)

best_val_mae = float("inf")
best_model_state = None
patience, patience_counter = 15, 0

for epoch in range(EPOCHS):
    if epoch == FREEZE_BACKBONE_EPOCHS:
        set_trainable(model.features, True)
        optimizer = torch.optim.AdamW([
            {"params": model.head.parameters(), "lr": LR},
            {"params": model.features.parameters(), "lr": LR / 10},
        ], weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - epoch, eta_min=1e-6)
        print(f"\n[Epoch {epoch}] Unfreezing backbone!")

    model.train()
    train_loss, train_mae, n = 0.0, 0.0, 0
    for imgs, target, raw_h in tqdm(train_loader, desc=f"Train Ep {epoch+1}", leave=False):
        imgs, target = imgs.to(DEVICE), target.float().to(DEVICE)
        optimizer.zero_grad()
        pred = model(imgs)
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(imgs)
        train_mae += torch.abs(pred - raw_h.to(DEVICE)).sum().item()
        n += len(imgs)

    model.eval()
    val_loss, val_mae, nv = 0.0, 0.0, 0
    with torch.no_grad():
        for imgs, target, raw_h in val_loader:
            imgs, target = imgs.to(DEVICE), target.float().to(DEVICE)
            pred = model(imgs)
            loss = criterion(pred, target)
            val_loss += loss.item() * len(imgs)
            val_mae += torch.abs(pred - raw_h.to(DEVICE)).sum().item()
            nv += len(imgs)

    train_loss /= n; train_mae /= n
    val_loss /= nv; val_mae /= nv
    scheduler.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Ep {epoch+1:3d} | Train Loss: {train_loss:.2f} | Train MAE: {train_mae:.2f}m | "
              f"Val Loss: {val_loss:.2f} | Val MAE: {val_mae:.2f}m | LR: {scheduler.get_last_lr()[0]:.6f}")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_counter = 0
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}. Best Val MAE: {best_val_mae:.2f}m")
        break

print(f"\nBest Validation MAE: {best_val_mae:.2f}m")

TRAINING


  Ep   1 | Train Loss: 34.12 | Train MAE: 12.73m | Val Loss: 15.53 | Val MAE: 6.51m | LR: 0.000100


  Ep   5 | Train Loss: 17.50 | Train MAE: 7.09m | Val Loss: 11.60 | Val MAE: 5.06m | LR: 0.000099

[Epoch 5] Unfreezing backbone!


  Ep  10 | Train Loss: 12.10 | Train MAE: 5.25m | Val Loss: 8.21 | Val MAE: 3.86m | LR: 0.000099


  Ep  15 | Train Loss: 9.84 | Train MAE: 4.47m | Val Loss: 7.16 | Val MAE: 3.52m | LR: 0.000096


...

  Ep  60 | Train Loss: 4.77 | Train MAE: 2.61m | Val Loss: 3.41 | Val MAE: 2.03m | LR: 0.000017


  Ep  65 | Train Loss: 4.49 | Train MAE: 2.51m | Val Loss: 3.20 | Val MAE: 1.96m | LR: 0.000010


  Ep  70 | Train Loss: 4.39 | Train MAE: 2.47m | Val Loss: 3.29 | Val MAE: 1.99m | LR: 0.000005


  Ep  75 | Train Loss: 4.23 | Train MAE: 2.41m | Val Loss: 3.28 | Val MAE: 1.98m | LR: 0.000002



Early stopping at epoch 78. Best Val MAE: 1.94m

Best Validation MAE: 1.94m


In [23]:
print("="*60 + "\nTEST SET EVALUATION\n" + "="*60)

model.load_state_dict(best_model_state)
model.eval()

preds_raw, targets_raw = [], []
with torch.no_grad():
    for imgs, _, raw_h in tqdm(test_loader, desc="Evaluating"):
        preds_raw.extend(model(imgs.to(DEVICE)).cpu().numpy().tolist())
        targets_raw.extend(raw_h.numpy().tolist())

P, T = np.array(preds_raw), np.array(targets_raw)

mae = float(np.mean(np.abs(P - T)))
rmse = float(np.sqrt(np.mean((P - T) ** 2)))
mape = float(np.mean(np.abs((P - T) / (T + 1e-6))) * 100)
r2 = float(1 - np.sum((T - P) ** 2) / np.sum((T - np.mean(T)) ** 2))

print(f"\n{'='*40}\nTEST RESULTS ({len(T)} samples)\n{'='*40}")
print(f"  MAE:  {mae:.3f} m")
print(f"  RMSE: {rmse:.3f} m")
print(f"  MAPE: {mape:.1f}%")
print(f"  R²:   {r2:.4f}")
print(f"  Pred: [{P.min():.1f}, {P.max():.1f}]m | True: [{T.min():.1f}, {T.max():.1f}]m")

print(f"\n  MAE by height bucket:")
for lo, hi in [(0, 5), (5, 10), (10, 20), (20, 50), (50, 200), (200, 1000)]:
    mask = (T >= lo) & (T < hi)
    if mask.sum() > 0:
        print(f"    {lo:4.0f}-{hi:4.0f}m: MAE = {np.mean(np.abs(P[mask] - T[mask])):.2f}m  (n={mask.sum()})")

torch.save(best_model_state, "convnext_height_model.pth")
print(f"\n{'='*50}\nFINAL ANSWER: MAE = {mae:.3f} meters\n{'='*50}")

TEST SET EVALUATION


Evaluating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19/19 [00:01<00:00, 13.41it/s]



TEST RESULTS (300 samples)
  MAE:  2.830 m
  RMSE: 5.925 m
  MAPE: 29.8%
  R²:   0.8874
  Pred: [2.6, 148.1]m | True: [2.5, 126.0]m

  MAE by height bucket:
       0-   5m: MAE = 1.45m  (n=102)
       5-  10m: MAE = 1.77m  (n=122)
      10-  20m: MAE = 3.92m  (n=20)
      20-  50m: MAE = 4.46m  (n=44)
      50- 200m: MAE = 17.56m  (n=12)

FINAL ANSWER: MAE = 2.830 meters
